In [4]:
%cd practicas/

[Errno 2] No such file or directory: 'practicas/'
/workspace/practicas


/usr/local/lib/python3.10/site-packages/IPython/core/magics/osm.py:393: UserWarning: This is now an optional IPython functionality, using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})


In [5]:
from pyspark.sql import SparkSession

spark = ( SparkSession.builder
         .appName("pr503")
         .master("spark://spark-master:7077")
         .getOrCreate()
         )
 
sc = spark.sparkContext

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/26 10:17:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
from pyspark.sql.types import StructType, StructField, BooleanType, IntegerType, StringType, DoubleType, LongType, TimestampType
from pyspark.sql import functions as f
from pyspark.sql import Window

schema_world = StructType([
    StructField("row_id", IntegerType(), True),
    StructField("click_datetime", TimestampType(), True),
    StructField("time_to_next_click", DoubleType(), True),
    StructField("movie_title", StringType(), True),
    StructField("movie_genres", StringType(), True),
    StructField("release_date", StringType(), True),
    StructField("title_id", StringType(), True),
    StructField("user_id", StringType(), True),
    ])

df = (spark.read
             .format("csv")
             .schema(schema_world)
             .option("header", "True")
             .load("./data/vodclickstream_uk_movies_03.csv"))
df.show(5)

+------+-------------------+------------------+--------------------+--------------------+------------+----------+----------+
|row_id|     click_datetime|time_to_next_click|         movie_title|        movie_genres|release_date|  title_id|   user_id|
+------+-------------------+------------------+--------------------+--------------------+------------+----------+----------+
| 58773|2017-01-01 01:15:09|               0.0|Angus, Thongs and...|Comedy, Drama, Ro...|  2008-07-25|26bd5987e8|1dea19f6fe|
| 58774|2017-01-01 13:56:02|               0.0|The Curse of Slee...|Fantasy, Horror, ...|  2016-06-02|f26ed2675e|544dcbc510|
| 58775|2017-01-01 15:17:47|           10530.0|   London Has Fallen|    Action, Thriller|  2016-03-04|f77e500e7a|7cbcc791bf|
| 58776|2017-01-01 16:04:13|              49.0|            Vendetta|       Action, Drama|  2015-06-12|c74aec7673|ebf43c36b6|
| 58777|2017-01-01 19:16:37|               0.0|The SpongeBob Squ...|Animation, Action...|  2004-11-19|a80d6fc2aa|a57c992287|


# 1. Auditoría de telemetría Web (validación de datos)

In [10]:
ventana = ( Window.partitionBy("user_id").orderBy("click_datetime") )

df_resultado = df.select("user_id", "click_datetime").withColumn(
    "click_anterior", f.lead("click_datetime", 1).over(ventana)
).withColumn("calculated_time_to_next", f.col("click_anterior") - f.col("click_datetime"))

df_resultado.show(5, truncate=False)

+----------+-------------------+-------------------+-----------------------------------+
|user_id   |click_datetime     |click_anterior     |calculated_time_to_next            |
+----------+-------------------+-------------------+-----------------------------------+
|0006ea6b5c|2017-05-19 20:21:43|2017-05-20 21:54:34|INTERVAL '1 01:32:51' DAY TO SECOND|
|0006ea6b5c|2017-05-20 21:54:34|2017-05-26 18:38:01|INTERVAL '5 20:43:27' DAY TO SECOND|
|0006ea6b5c|2017-05-26 18:38:01|2017-05-26 23:31:46|INTERVAL '0 04:53:45' DAY TO SECOND|
|0006ea6b5c|2017-05-26 23:31:46|2017-05-27 22:45:41|INTERVAL '0 23:13:55' DAY TO SECOND|
|0006ea6b5c|2017-05-27 22:45:41|2017-06-02 22:51:18|INTERVAL '6 00:05:37' DAY TO SECOND|
+----------+-------------------+-------------------+-----------------------------------+
only showing top 5 rows

